In [1]:
!pip install pyreadr

In [2]:
import os.path
import pyreadr
import glob
import pandas as pd
import re

/Users/xzhao17/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/xzhao17/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/Users/xzhao17/opt/anaconda3/lib/python3.9/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [9]:
#DATA_DIR = "../hometimefinaldata"
# DATA_DIR = "../march7result"
#DATA_DIR = "./new/"
DATA_DIR = "../june10result/"
"""
How STUDY_TIME is calculated
----------------------------
1. Glob all files and folders inside DATA_DIR
2. For each path found:
3.     Take the name of the folder using os.path.basename (removing the long path name and taking only the name of the folder)
4.     Remove the "days" part from the folder name by using [:-4]
5.     Convert the remaining string to integer using int()
"""
STUDY_TIME = [int(os.path.basename(path)[:-4]) for path in glob.glob(DATA_DIR + "/*")]
STUDY_TIME

[365]

In [10]:
#365
# this_st = STUDY_TIME[1]
#90
this_st = STUDY_TIME[0]

TYPE_I_DATA_FILES = DATA_DIR + f"/{this_st}days/TypeI*"

glob.glob(TYPE_I_DATA_FILES)

['../june10result//365days/TypeI_SmallDiffCensor_Balance_NoCensor365.RData',
 '../june10result//365days/TypeI_UniformCensor_Unbalance_NoCensor365.RData',
 '../june10result//365days/TypeI_ReverseBigDiffCensor_Unbalance_NoCensor365.RData',
 '../june10result//365days/TypeI_ReverseBigDiffCensor_Balance_NoCensor365.RData',
 '../june10result//365days/TypeI_Uncensor_Balance_NoCensor365.RData',
 '../june10result//365days/TypeI_Uncensor_Unbalance_NoCensor365.RData',
 '../june10result//365days/TypeI_ReverseSmallDiffCensor_Unbalance_NoCensor365.RData',
 '../june10result//365days/TypeI_BigDiffCensor_Balance_NoCensor365.RData',
 '../june10result//365days/TypeI_BigDiffCensor_Unbalance_NoCensor365.RData',
 '../june10result//365days/TypeI_UniformCensor_Balance_NoCensor365.RData',
 '../june10result//365days/TypeI_SmallDiffCensor_Unbalance_NoCensor365.RData',
 '../june10result//365days/TypeI_ReverseSmallDiffCensor_Balance_NoCensor365.RData']

In [11]:
FILE_ATTRIBUTES = { os.path.basename(path): os.path.basename(path)[6:-6].split("_") for path in glob.glob(TYPE_I_DATA_FILES) }
FILE_ATTRIBUTES

{'TypeI_SmallDiffCensor_Balance_NoCensor365.RData': ['SmallDiffCensor',
  'Balance',
  'NoCensor365'],
 'TypeI_UniformCensor_Unbalance_NoCensor365.RData': ['UniformCensor',
  'Unbalance',
  'NoCensor365'],
 'TypeI_ReverseBigDiffCensor_Unbalance_NoCensor365.RData': ['ReverseBigDiffCensor',
  'Unbalance',
  'NoCensor365'],
 'TypeI_ReverseBigDiffCensor_Balance_NoCensor365.RData': ['ReverseBigDiffCensor',
  'Balance',
  'NoCensor365'],
 'TypeI_Uncensor_Balance_NoCensor365.RData': ['Uncensor',
  'Balance',
  'NoCensor365'],
 'TypeI_Uncensor_Unbalance_NoCensor365.RData': ['Uncensor',
  'Unbalance',
  'NoCensor365'],
 'TypeI_ReverseSmallDiffCensor_Unbalance_NoCensor365.RData': ['ReverseSmallDiffCensor',
  'Unbalance',
  'NoCensor365'],
 'TypeI_BigDiffCensor_Balance_NoCensor365.RData': ['BigDiffCensor',
  'Balance',
  'NoCensor365'],
 'TypeI_BigDiffCensor_Unbalance_NoCensor365.RData': ['BigDiffCensor',
  'Unbalance',
  'NoCensor365'],
 'TypeI_UniformCensor_Balance_NoCensor365.RData': ['Uniform

In [12]:
def read_one_variable_RData(file_path):
    type1 = pyreadr.read_r(file_path)
    return type1[ list(type1.keys())[0] ]

In [13]:
COLUMN_NAMES = {"lin": "Linear", "med": "Median", "poi": "Poisson", "nb": "Neg Bin", "cox":"Cox PH", "temp":"Temp Pro"}

In [14]:
EMPTY_LINE = '\\multicolumn{1}{l|}{} & \\multicolumn{1}{l|}{} & & & \\multicolumn{1}{l|}{} & & & \\\\'
EMPTY_LINE

'\\multicolumn{1}{l|}{} & \\multicolumn{1}{l|}{} & & & \\multicolumn{1}{l|}{} & & & \\\\'

In [15]:
PRE_TEXT = r"""\begin{table}[H]
\resizebox{\textwidth}{!}{
\begin{tabular}{llllllll}
                                           &                                       & \multicolumn{3}{c}{No Treatment Effect ($\rho=0$)}   & \multicolumn{3}{c}{Treatment Effect ($\rho =1$)} \\ \cline{3-8} 
n                                          & Methods                               & Estimate & SE ($\hat{\rho})$ & \multicolumn{1}{l|}{Type I Error Rate$^1$ (SE)} & Estimate           & Standard Error           & Power$^1 $(SE)          \\ \hline"""

print(PRE_TEXT)

\begin{table}[H]
\resizebox{\textwidth}{!}{
\begin{tabular}{llllllll}
                                           &                                       & \multicolumn{3}{c}{No Treatment Effect ($\rho=0$)}   & \multicolumn{3}{c}{Treatment Effect ($\rho =1$)} \\ \cline{3-8} 
n                                          & Methods                               & Estimate & SE ($\hat{\rho})$ & \multicolumn{1}{l|}{Type I Error Rate$^1$ (SE)} & Estimate           & Standard Error           & Power$^1 $(SE)          \\ \hline


In [16]:
POST_TEXT = r"""\bottomrule
\multicolumn{8}{l}{\small $^1$ $\alpha=0.05$} \\
\end{tabular}
}
\caption{ <caption/> }
\label{alpha365b}
\end{table}"""

print(POST_TEXT)

\bottomrule
\multicolumn{8}{l}{\small $^1$ $\alpha=0.05$} \\
\end{tabular}
}
\caption{ <caption/> }
\label{alpha365b}
\end{table}


In [17]:
POST_TEXT = r"""\bottomrule
\multicolumn{8}{l}{\small $^1$ $\alpha=0.05$} \\
\end{tabular}
}
\caption{ <caption/> }
\label{alpha90b}
\end{table}"""

print(POST_TEXT)

\bottomrule
\multicolumn{8}{l}{\small $^1$ $\alpha=0.05$} \\
\end{tabular}
}
\caption{ <caption/> }
\label{alpha90b}
\end{table}


In [18]:
re.sub("<caption/>", "HELLO", POST_TEXT)

'\\bottomrule\n\\multicolumn{8}{l}{\\small $^1$ $\\alpha=0.05$} \\\\\n\\end{tabular}\n}\n\\caption{ HELLO }\n\\label{alpha90b}\n\\end{table}'

In [19]:
# type1_file = f"{DATA_DIR}/{this_st}days/typeI_{censoring}_{balance}.RData"

In [20]:
list(FILE_ATTRIBUTES.values())[0]

['SmallDiffCensor', 'Balance', 'NoCensor365']

In [21]:
def write_table(type1_error_rate, power_result):
    for offset in [0, 5, 10]:
        if offset != 0:
            print(EMPTY_LINE)
        print("%================================")
        for idx in range(type1_error_rate.shape[0]):
            print("%", offset, type1_error_rate.iloc[idx].name, "\n%")
            tline = type1_error_rate.iloc[idx]
            pline = power_result.iloc[idx]
            col_name = COLUMN_NAMES[type1_error_rate.iloc[idx].name]
            num_indvs = {0: 500, 5: 1000, 10: 5000}[offset]
            multi_line_num_indv_str = f"\multirow{{6}}{{*}}{{{num_indvs}}}" if idx == 0 else ""
            short = f"\multicolumn{{1}}{{l|}}{{{multi_line_num_indv_str}}} & \multicolumn{{1}}{{l|}}{{{col_name}}} & \multicolumn{{1}}{{c}}{{{tline[offset + 0]:.2f} ({pline[offset + 1]:.2f})}} & \multicolumn{{1}}{{c}} {{{tline[offset + 2]:.2f}}} & \multicolumn{{1}}{{c|}}{{{tline[offset + 3]:.2f} ({tline[offset + 4]:.2f})}} & \multicolumn{{1}}{{c}}{{{pline[offset + 0]:.2f} ({pline[offset + 1]:.2f})}} & \multicolumn{{1}}{{c}}{{{pline[offset + 2]:.2f}}} & \multicolumn{{1}}{{c}}{{{pline[offset + 3]:.2f} ({pline[offset + 4]:.2f})}} \\\\"
            print(short)
            print("%----")

In [16]:
for censoring, balance, no365 in FILE_ATTRIBUTES.values():
    type1_file = f"{DATA_DIR}/{this_st}days/TypeI_{censoring}_{balance}_{no365}.RData"
    power_file = f"{DATA_DIR}/{this_st}days/Power_{censoring}_{balance}_{no365}.RData"
    type1_error_rate = read_one_variable_RData(type1_file)
    power_result = read_one_variable_RData(power_file)
    print(PRE_TEXT)
    write_table(type1_error_rate, power_result)
    post_text = re.sub("<caption/>", f"{censoring} {balance} {no365}", POST_TEXT)
    print(post_text)
    print("\n\n\n")
    

\begin{table}[H]
\resizebox{\textwidth}{!}{
\begin{tabular}{llllllll}
                                           &                                       & \multicolumn{3}{c}{No Treatment Effect ($\rho=0$)}   & \multicolumn{3}{c}{Treatment Effect ($\rho =1$)} \\ \cline{3-8} 
n                                          & Methods                               & Estimate & SE ($\hat{\rho})$ & \multicolumn{1}{l|}{Type I Error Rate$^1$ (SE)} & Estimate           & Standard Error           & Power$^1 $(SE)          \\ \hline
%================================
% 0 lin 
%
\multicolumn{1}{l|}{\multirow{6}{*}{500}} & \multicolumn{1}{l|}{Linear} & \multicolumn{1}{c}{-2.40 (13.34)} & \multicolumn{1}{c} {20.38} & \multicolumn{1}{c|}{0.20 (0.18)} & \multicolumn{1}{c}{11.31 (13.34)} & \multicolumn{1}{c}{21.93} & \multicolumn{1}{c}{0.20 (0.18)} \\
%----
% 0 med 
%
\multicolumn{1}{l|}{} & \multicolumn{1}{l|}{Median} & \multicolumn{1}{c}{2.94 (27.73)} & \multicolumn{1}{c} {71.66} & \multicolumn{1}{c|}{0.40

In [17]:
type1_error_rate

,est,semodel,se,power,powerse,est,semodel,se,power,powerse,est,semodel,se,power,powerse
lin,-1.957975,11.095936,14.567217,0.2,0.178885,1.797546,7.800919,6.631220,0.0,0.0,1.599221,3.515095,4.044831,0.2,0.178885
med,-3.798156,17.342739,14.510698,0.0,0.000000,4.312665,11.568028,5.741819,0.0,0.0,1.161121,4.422912,3.692803,0.0,0.000000
poi,-0.011543,0.062245,0.081408,0.2,0.178885,0.009959,0.044080,0.037351,0.0,0.0,0.009015,0.019731,0.022746,0.2,0.178885
nb,-0.011543,0.062647,0.081408,0.2,0.178885,0.009959,0.044091,0.037351,0.0,0.0,0.009015,0.019697,0.022746,0.2,0.178885
cox,0.033392,0.129432,0.107844,0.0,0.000000,-0.011364,0.089342,0.095721,0.0,0.0,-0.017366,0.040214,0.039776,0.0,0.000000
temp,1.971922,13.244762,16.919749,0.2,0.178885,-1.818128,9.265919,8.342247,0.0,0.0,-1.822818,4.182015,4.982008,0.2,0.178885


In [18]:
power_result

,est,semodel,se,power,powerse,est,semodel,se,power,powerse,est,semodel,se,power,powerse
lin,6.283369,11.029026,14.786471,0.2,0.178885,9.755134,7.739605,6.345559,0.2,0.178885,9.755115,3.494946,3.895762,0.8,0.178885
med,-1.051548,17.247113,12.735359,0.0,0.000000,7.297709,11.486076,6.967913,0.0,0.000000,5.342824,4.397899,4.173348,0.2,0.178885
poi,0.033952,0.060906,0.080446,0.2,0.178885,0.053818,0.043081,0.034811,0.2,0.178885,0.053543,0.019329,0.021572,0.8,0.178885
nb,0.033952,0.060595,0.080446,0.2,0.178885,0.053818,0.042665,0.034811,0.2,0.178885,0.053543,0.019085,0.021572,0.8,0.178885
cox,-0.210996,0.133287,0.149628,0.4,0.219089,-0.271553,0.092767,0.093527,1.0,0.000000,-0.272179,0.041692,0.038354,1.0,0.000000
temp,-9.244622,13.393068,17.634625,0.2,0.178885,-12.621736,9.370456,7.900811,0.2,0.178885,-13.200625,4.234245,4.827378,0.8,0.178885


In [78]:
type1_error_rate = read_one_variable_RData( DATA_DIR + f"/{this_st}days/typeI_4censor_unbalance.RData" )
type1_error_rate

,est,se,power,powerse,est,se,power,powerse,est,se,power,powerse
lin,0.362302,0.603238,0.0518,0.003134,0.017892,0.296549,0.0482,0.003029,0.079355,0.059787,0.0504,0.003094
med,2.627899,1.499428,0.1784,0.005414,0.791069,0.743953,0.2078,0.005738,0.417621,0.150005,0.2052,0.005711
poi,0.002206,0.002982,0.0488,0.003047,0.000305,0.001465,0.0480,0.003023,0.000437,0.000295,0.0514,0.003123
nb,0.002206,0.002982,0.0516,0.003128,0.000305,0.001465,0.0484,0.003035,0.000437,0.000295,0.0504,0.003094
cox,-0.003052,0.004242,0.0484,0.003035,-0.000548,0.002134,0.0486,0.003041,-0.000665,0.000428,0.0506,0.003100
temp,-0.375971,0.626823,0.0496,0.003070,-0.056359,0.306386,0.0506,0.003100,-0.109710,0.062160,0.0526,0.003157


In [79]:
power_result = read_one_variable_RData( DATA_DIR + f"/{this_st}days/power_scensor_balance.RData" )
power_result

,est,se,power,powerse,est,se,power,powerse,est,se,power,powerse
lin,14.762537,0.595049,0.1970,0.005625,14.900744,0.301903,0.3578,0.006779,14.777787,0.058529,0.9472,0.003163
med,33.268727,1.821170,0.4636,0.007052,35.344139,0.947640,0.6456,0.006765,36.947404,0.188561,0.9558,0.002907
poi,0.071044,0.002866,0.1980,0.005636,0.071629,0.001453,0.3586,0.006782,0.070995,0.000281,0.9474,0.003157
nb,0.071044,0.002866,0.1950,0.005603,0.071629,0.001453,0.3560,0.006771,0.070995,0.000281,0.9472,0.003163
cox,-0.469173,0.004374,0.9990,0.000447,-0.468952,0.002230,1.0000,0.000000,-0.466970,0.000439,1.0000,0.000000
temp,-17.238860,0.617691,0.2350,0.005996,-17.322398,0.314753,0.4246,0.006990,-17.198165,0.061493,0.9750,0.002208


In [111]:
mystr = r"\multicolumn{1}{l|}{}                      & \multicolumn{1}{l|}{}                 &          &                & \multicolumn{1}{l|}{}                  &                    &                          &                \\"
mystr = re.sub(r"\s+", " ", mystr)
# mystr = re.sub(r"{", "{{", mystr)
# mystr = re.sub(r"}", "}}", mystr)
print(mystr)
mystr

\multicolumn{1}{l|}{} & \multicolumn{1}{l|}{} & & & \multicolumn{1}{l|}{} & & & \\


'\\multicolumn{1}{l|}{} & \\multicolumn{1}{l|}{} & & & \\multicolumn{1}{l|}{} & & & \\\\'